# Mobile Price Range Classification

**Classification Project 20 of 100** - Predict the price range of a mobile phone from its specifications.

End-to-end workflow: EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

This notebook builds a 4-class classifier predicting a phone price range (0=low to 3=very high) from 20 hardware specifications. Classes are perfectly balanced (500 each), all features numeric.

## 3. Learning Objectives

1. Perfectly balanced 4-class dataset (2000 rows)
2. All-numeric features (20 phone specs)
3. Understand which specs drive price
4. Evaluate with macro-F1, accuracy, confusion matrix
5. Benchmark with LazyPredict and FLAML
6. Learn that RAM is the strongest price predictor
7. Full benchmark -> AutoML -> top 3 pipeline

## 4. Problem Statement

> Given 20 mobile specs (RAM, battery, screen, camera, etc.), predict price range: 0 (low) to 3 (very high).

4-class classification with balanced classes. Macro-F1 and accuracy are appropriate.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Manufacturers | Competitive pricing |
| E-commerce | Auto-categorisation |
| Consumers | Price-feature understanding |
| ML learners | Clean multiclass with numeric features |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | iabhishekofficial/mobile-price-classification |
| Rows | 2,000 |
| Features | 20 numeric |
| Target | price_range (0,1,2,3) |
| Class balance | Perfectly balanced (500 each) |

Key: ram (~0.92 correlation), battery_power, px_height, px_width

## 7. Dataset Source and License Notes

- Kaggle: https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification
- License: CC0 Public Domain.
- Synthetic with realistic distributions.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "iabhishekofficial/mobile-price-classification"
TARGET = "price_range"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120
print(f"Target: {TARGET}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
train_csv = [f for f in csv_files if "train" in os.path.basename(f).lower()]
df_raw = pd.read_csv(train_csv[0] if train_csv else csv_files[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
print(f"Missing: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
assert TARGET in df.columns
print(f"\nTarget:\n{df[TARGET].value_counts().sort_index()}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
df[TARGET].value_counts().sort_index().plot.bar(ax=ax, color=sns.color_palette("Set2", 4))
ax.set_title("Price Range Distribution")
ax.set_xticklabels(["Low","Medium","High","Very High"], rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
key_feats = [c for c in ["ram","battery_power","px_height","px_width","int_memory"] if c in df.columns]
fig, axes = plt.subplots(1, len(key_feats), figsize=(5*len(key_feats), 4))
if len(key_feats)==1: axes=[axes]
for ax, col in zip(axes, key_feats):
    for pr in sorted(df[TARGET].unique()):
        ax.hist(df[df[TARGET]==pr][col], bins=25, alpha=0.4, label=f"Range {pr}")
    ax.set_title(col); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
corr = df.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Top 10 correlations:")
print(corr.head(10))

## 14. Target Analysis

4 perfectly balanced classes. RAM has ~0.92 correlation with price_range.

In [ ]:
print(f"Class balance: {df[TARGET].value_counts(normalize=True).to_dict()}")
print(f"Majority baseline: {df[TARGET].value_counts(normalize=True).max():.1%}")

## 15. Train / Validation / Test Split

In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## 16. Preprocessing Strategy

All numeric, no missing values. StandardScaler.

In [ ]:
num_cols = X_train.columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols)
], remainder="passthrough")
print(f"Features: {len(num_cols)}")

## 17. Feature Engineering

- SCREEN_AREA: px_height * px_width
- RAM_PER_CORE: ram / n_cores
- BATTERY_PER_WEIGHT: battery_power / mobile_wt

In [ ]:
def eng(d):
    d = d.copy()
    if "px_height" in d.columns and "px_width" in d.columns:
        d["SCREEN_AREA"] = d["px_height"] * d["px_width"]
    if "ram" in d.columns and "n_cores" in d.columns:
        d["RAM_PER_CORE"] = d["ram"] / d["n_cores"].clip(lower=1)
    if "battery_power" in d.columns and "mobile_wt" in d.columns:
        d["BATTERY_PER_WEIGHT"] = d["battery_power"] / d["mobile_wt"].clip(lower=1)
    return d
X_train=eng(X_train); X_val=eng(X_val); X_test=eng(X_test)
num_cols = X_train.columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols)
], remainder="passthrough")
print(f"Features: {X_train.shape[1]}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("LogReg",LogisticRegression(max_iter=1000,random_state=SEED)),
                  ("RF",RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp=pipe.predict(X_val)
    r={"Accuracy":accuracy_score(y_val,yp),"Macro-F1":f1_score(y_val,yp,average="macro")}
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp=preprocessor.fit_transform(X_train); X_val_lp=preprocessor.transform(X_val)
lazy=LazyClassifier(verbose=0,ignore_warnings=True,random_state=SEED)
models_lp,_=lazy.fit(X_train_lp,X_val_lp,y_train,y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl=AutoML()
automl.fit(X_train, y_train, task="classification", metric="macro_f1", time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_val)
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"Macro-F1":f1_score(y_val,yp_fl,average="macro")}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
top3={}
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=500,random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="mlogloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"]=AdaBoostClassifier(n_estimators=200,random_state=SEED)
top3["GradBoosting"]=GradientBoostingClassifier(n_estimators=300,learning_rate=0.05,max_depth=5,random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t=preprocessor.fit_transform(X_train); X_te_t=preprocessor.transform(X_test)
labels = ["Low","Medium","High","Very High"]
final={}
for name,mdl in top3.items():
    mdl.fit(X_tr_t,y_train); yp=mdl.predict(X_te_t)
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Macro-F1":f1_score(y_test,yp,average="macro")}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=labels))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_te_t),ax=ax,cmap="Blues",display_labels=labels); ax.set_title(n)
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_te_t)
misclass = (y_test.values != ypb).sum()
print(f"Misclassified: {misclass}/{len(y_test)}")
if misclass > 0:
    idx = np.where(y_test.values != ypb)[0]
    td = X_test.iloc[idx].copy()
    td["true"] = y_test.values[idx]; td["pred"] = ypb[idx]
    if "ram" in td.columns: print(f"Mean RAM (misclassified): {td['ram'].mean():.0f} vs overall: {X_test['ram'].mean():.0f}")

## 24. Interpretation and Business Insight

**Key findings:**
1. RAM has ~0.92 correlation with price range
2. Battery power and pixel dimensions are secondary
3. Errors occur at class boundaries
4. Even LogReg achieves >90% accuracy

RAM is the primary driver of phone value perception.

## 25. Limitations

1. Synthetic data (no brand, OS, design)
2. No brand feature
3. RAM dominance (real pricing more nuanced)
4. 2000 rows
5. Static (pricing evolves rapidly)

## 26. How to Improve This Project

1. Cross-validation
2. LogReg with only RAM
3. Interaction features
4. SHAP analysis
5. SVM comparison
6. Ordinal classification

## 27. Production Considerations

- E-commerce auto-categorisation
- RAM thresholds shift annually
- Regional pricing differs
- Brand factor needed for real use
- Annual retraining

## 28. Common Mistakes

1. Not noticing RAM dominance
2. Overcomplicating the model
3. Not using stratified splits
4. Ignoring ordinal structure
5. Treating as regression (valid alternative)

## 29. Mini Challenge / Exercises

1. Train with only RAM - what accuracy?
2. Find RAM thresholds between price ranges
3. Ordinal regression - does order help?
4. SHAP vs correlation rankings
5. Which adjacent classes most confused?

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | 4-class classification - mobile price |
| Dataset | 2,000 phones, 20 features, balanced |
| Difficulty | Easy - strong RAM correlation |
| Best metric | Macro-F1 and accuracy |
| Baseline | 25% |
| Key insight | RAM explains ~90% of price |
| Limitation | Synthetic, no brand |

**What you learned:**
- Multiclass with balanced classes
- Feature importance and dominant predictors
- Simple models match complex when signal is strong
- Full benchmark -> AutoML -> top 3 pipeline